سنرى هنا تقنية أخرى لتقليل البعد الثنائي تسمى LDA ونقارنها مع PCA. في وقت لاحق، باستخدام هذا سوف نقوم بنمذجة الموضوع.
* إل دي إيه
* LDA مقابل PCA
* نمذجة الموضوع


In [ ]:
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.stem.porter import *
from nltk.tokenize import word_tokenize
from sklearn.datasets import fetch_20newsgroups, load_iris
from sklearn.decomposition import PCA, LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer


### LDA (التحليل التمييزي الخطي) وLDA مقابل PCA
**LDA** هي تقنية لتقليل الأبعاد. تعمل على تحويل بياناتك من أبعاد 'n' إلى أبعاد 'k'.   كلاهما متشابهان إلى حد كبير في الإخراج ولكن مع اختلاف **كبير** واحد. LDA عبارة عن خوارزمية خاضعة للإشراف بينما PCA ليست كذلك، ويتجاهل PCA **تصنيفات الفئة**.
 كما رأينا في الأسابيع السابقة، يحاول PCA العثور على اتجاهات الحد الأقصى من التباين. تقوم PCA بإسقاط البيانات على محور جديد بطريقة تشرح الحد الأقصى من التباين دون أخذ تسميات الفئة بعين الاعتبار. **LDA** من ناحية أخرى، تقوم بإنشاء محور جديد بطريقة أنه عندما نقوم بإسقاط البيانات على هذا المحور، يكون هناك حد أقصى للفصل بين فئتين فئتين. يحاول LDA فصل الفئات قدر الإمكان على المحور الجديد. 
 
فيما يلي عرض توضيحي للشيء نفسه مع مجموعة بيانات Iris.  


In [ ]:
data=load_iris().data
target=load_iris().target
target_names=load_iris().target_names

In [ ]:
dataframe=pd.DataFrame(data=np.concatenate((data,target.reshape(150,1)),axis=1),columns=['col_1','col_2','col_3','col_4','target'])

In [ ]:
dataframe.head()

In [ ]:
dataframe.drop(columns=['target'],axis=1,inplace=True)

In [ ]:
pca = PCA (n_components=2)
X_feature_reduced = pca.fit(dataframe).transform(dataframe)

In [ ]:
plt.scatter(X_feature_reduced[:,0],X_feature_reduced[:,1],c=target)
plt.title("PCA")
plt.show()

In [ ]:
 lda = LatentDirichletAllocation(n_components=2)

In [ ]:
X_feature_reduced = lda.fit(dataframe).transform(dataframe)

In [ ]:
plt.scatter(X_feature_reduced[:,0],X_feature_reduced[:,1],c=target)
plt.title('LDA')
plt.show()


### الملاحظة
كما يمكننا أن نرى من الأعلى أن بيانات LDA المتوقعة على المحور الجديد بطريقة يتم فصل الفصل قدر الإمكان. 



#### كيف تحقق LDA ذلك؟
تقوم LDA بإنشاء محور جديد بناءً على معيارين:
* المسافة بين وسائل الطبقات
*التنوع داخل كل فئة
يقوم بإسقاط البيانات على محور جديد ويجد المتوسط لكل فئة والتباين لكل فئة. يحاول تعظيم المسافة بين وسائل الفصل ويحاول تقليل الاختلاف مع كل فئة. باستخدام هذه في الاعتبار نحصل على محور جديد.



![نص بديل](../../img/shivam_panwar_data1.png "data")
أعلاه هي البيانات الخاصة بجينين، ونريد إسقاطهما على محور جديد ببعد واحد.



<img src="../../img/shivam_panwar_transformed_data.png">


المعيار الذي نختاره أعلاه لحل هذه المشكلة هو
**(μ1-μ2)^2
/(س1+س2)^2**
، حيث μ1 و μ2 متوسطان لكل فئة و 's1 و s2' عبارة عن تباين/مبعثر داخل فئة أثناء إنشاء محور جديد.
نحن نحاول تعظيم هذه المعايير أثناء إنشاء محور جديد.



### نمذجة الموضوع 
**نمذجة الموضوع** هي طريقة لتعيين موضوع لكل مستند. كل موضوع يتكون من كلمات معينة.
خذ بعين الاعتبار على سبيل المثال:
لدينا موضوعان، الموضوع 1 والموضوع 2. **'الموضوع 1'** يتم تمثيله بواسطة "التفاح والموز والمانجو" و**الموضوع 2** يتم تمثيله بواسطة "التنس والكريكيت والهوكي". يمكننا أن نستنتج أن الموضوع 1 يتحدث عن الفواكه والموضوع 2 يتحدث عن الرياضة. يمكننا تعيين مستند وارد جديد في أحد هذه المواضيع ويمكن استخدامه لغرض **التجميع** أيضًا. يتم استخدامه في أنظمة التوصية وغيرها الكثير. 
مثال آخر: 
اعتبر أن لدينا 6 وثائق
* تفاح موز
* برتقال تفاح
* برتقال موز
* قطة النمر
* كلب النمر
* كلب قطة
ما ستفعله نمذجة الموضوع هو إذا أردت استخراج موضوعين من هذه المستندات، على سبيل المثال، فإنها ستوفر توزيعين، توزيع كلمات الموضوع وتوزيع موضوع المستند.  في تمثيل الكلمات الموضوعية، يجب أن يعطي توزيعًا حكيمًا للكلمات لكل موضوع وفي doc-topic سيعطي لكل مستند، فهو تمثيل الموضوع أو توزيع المستند لكل موضوع.
يجب أن يكون التوزيع المثالي لكلمات الموضوع:
|  الموضوع | أبل | موز | برتقالي | النمر | قطة | كلب | 
| --- | --- | --- | --- | --- | --- | --- | 
| الموضوع 1 |   .33 | .33 | .33 | 0 | 0 | 0 |
| الموضوع 2 |   0 | 0 | 0 | 0.33 | 0.33 | 0.33 |
ويجب أن يكون التوزيع المثالي لموضوع المستند هو:
|  الموضوع | وثيقة 1 | وثيقة2 ​​| وثيقة 3 | وثيقة 4 | وثيقة 5 | وثيقة 6 | 
| --- | --- | --- | --- | --- | --- | --- | 
| الموضوع 1 |   1 | 1 | 1 | 0 | 0 | 0 |
| الموضوع 2 |   0 | 0 | 0 | 1 | 1 | 1 |والآن لنفترض أن لدينا مستندًا جديدًا يقول، "تفاحة قطة كلب"، يجب أن يكون تمثيل موضوعها حكيمًا
**الموضوع1: 0.33**
**الموضوع2: 0.63**
يتم استخدام LDA بشكل كبير لهذا الغرض. إنه يستخدم لنمذجة الموضوع وقد تم توضيحه أدناه. نعطيها عدد المواضيع التي نريد معرفتها من المجموعة. تذكر أنه اتبع نهج القوس وبالتالي، يتم فقدان العلاقة بين الكلمات بهذه الطريقة.


In [ ]:
lemmatizer=WordNetLemmatizer() #For words Lemmatization
stemmer=PorterStemmer()  #For stemming words
stop_words=set(stopwords.words('english'))

In [ ]:
def TokenizeText(text):
    ''' 
     Tokenizes text by removing various stopwords and lemmatizing them
    '''
    text=re.sub('[^A-Za-z0-9\s]+', '', text)
    word_list=word_tokenize(text)
    word_list_final=[]
    for word in word_list:
        if word not in stop_words:
            word_list_final.append(lemmatizer.lemmatize(word))
    return word_list_final

In [ ]:
def gettopicwords(topics,cv,n_words=10):
    '''
        Print top n_words for each topic.
        cv=Countvectorizer
    '''
    for i,topic in enumerate(topics):
        top_words_array=np.array(cv.get_feature_names())[np.argsort(topic)[::-1][:n_words]]
        print "For  topic {} it's top {} words are ".format(str(i),str(n_words))
        combined_sentence=""
        for word in top_words_array:
            combined_sentence+=word+" "
        print combined_sentence
        print " "

In [ ]:
df=pd.read_csv('million-headlines.zip',usecols=[1])


رابط البيانات: 
[https://www.kaggle.com/therohk/million-headlines](https://www.kaggle.com/therohk/million-headlines)


In [ ]:
df.head()

In [ ]:
%%time
num_features=100000
# cv=CountVectorizer(min_df=0.01,max_df=0.97,tokenizer=TokenizeText,max_features=num_features)
cv=CountVectorizer(tokenizer=TokenizeText,max_features=num_features)
transformed_data=cv.fit_transform(df['headline_text'])

In [ ]:
transformed_data

In [ ]:
%%time
no_topics=10  ## We can change this, hyperparameter
lda = LatentDirichletAllocation(n_components=no_topics, max_iter=5, learning_method='online', learning_offset=50.,random_state=0).fit(transformed_data)


**Lda.components_** عبارة عن جدول كلمات_موضوع، يعرض تمثيل كل كلمة في الموضوع. يمكن عرض المكونات_[i, j] كعدد زائف يمثل عدد المرات التي تم فيها تخصيص الكلمة j للموضوع i. ويمكن أيضًا اعتباره توزيعًا على الكلمات لكل موضوع بعد التطبيع


In [ ]:
gettopicwords(lda.components_,cv)


### تعيين موضوع جديد 



يمكننا أن نرى أن كل وثيقة هي مزيج من كل موضوع. دعونا نرى تمثيل الموضوع للوثائق العشرة الأولى.



يتم عرض الوثائق العشرة الأولى وتمثيلها الموضوعي أدناه


In [ ]:
docs=df['headline_text'][:10]

In [ ]:
data=[]
for doc in docs:
    data.append(lda.transform(cv.transform([doc])))

In [ ]:
cols=['topic'+str(i) for i in range(1,11)]
doc_topic_df=pd.DataFrame(columns=cols,data=np.array(data).reshape((10,10)))

In [ ]:
doc_topic_df['major_topic']=doc_topic_df.idxmax(axis=1)
doc_topic_df['raw_doc']=docs

In [ ]:
doc_topic_df


لقد رأينا كيف يمكن استخدام LDA لنمذجة الموضوع. يمكن استخدام هذا في تجميع المستندات استنادًا إلى تمثيل موضوع المستند. 



### المراجع
[Statquest LDA](https://www.youtube.com/watch?v=azXCzI57Yfc)
[https://www.analyticsvidhya.com/blog/2016/08/beginners-guide-to-topic-modeling-in-python/](https://www.analyticsvidhya.com/blog/2016/08/beginners-guide-to-topic-modeling-in-python/)
[https://sebastianraschka.com/faq/docs/lda-vs-pca.html](https://sebastianraschka.com/faq/docs/lda-vs-pca.html)